<a href="https://colab.research.google.com/github/tsm-mehmetakiftasoz/tsm_makif/blob/main/lotlama_self_learning_by_words.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import pandas as pd

file_path = "/content/me5a.csv"

try:
    me5a = pd.read_csv(file_path, sep=",", encoding="utf-8-sig")
except:
    try:
        me5a = pd.read_csv(file_path, sep=";", encoding="utf-8-sig")
    except:
        me5a = pd.read_csv(file_path, engine="python")

print(me5a.shape)
me5a.head()

(205143, 12)


,ГОД,Заявка,Позиция заявки,Материал,Индикатор удаления_B3,Группа деблокир.,Дата заявки,RU Наименование,Дополнительное,Инвентарный номер,Закуп. организация,Группа материалов
0,2025,2200017176,10,4.500520e+09,False,2.0,7.12.2025,"1.3 ИЗДЕЛИЯ ТРУБ-ДОВ P <= 2.5 МРАG, 4 КЛ. БЕЗ....",NaN,AKU.0120.10UKC.QCA.TM.TB0016(C01),2992,MZ-2065
1,2025,2200017176,20,4.500520e+09,False,2.0,7.12.2025,"1.4 ИЗДЕЛИЯ ТРУБ-ДОВ P <= 2.5 МРАG, 4 КЛ. БЕЗ....",NaN,AKU.0120.10UKC.QCA.TM.TB0016(C01),2992,MZ-2065
2,2025,2200017177,10,4.500520e+09,False,2.0,7.12.2025,"1.3 ИЗДЕЛИЯ ТРУБ-ДОВ P <= 2.5 МРАG, 4 КЛ. БЕЗ....",NaN,СПЕЦИФИКАЦИИ JV,2992,MZ-2065
3,2025,2200017177,20,4.500520e+09,False,2.0,7.12.2025,"1.4 ИЗДЕЛИЯ ТРУБ-ДОВ P <= 2.5 МРАG, 4 КЛ. БЕЗ....",NaN,СПЕЦИФИКАЦИИ JV,2992,MZ-2065
4,2025,2200017178,10,4.500520e+09,False,2.0,7.12.2025,"1.2 ИЗДЕЛИЯ ТРУБ-ДОВ НИЗКОГО ДАВЛЕНИЯ, 4 КЛ. Б...",NaN,СПЕЦИФИКАЦИИ JV,2992,MZ-2065


In [19]:
me5a.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205143 entries, 0 to 205142
Data columns (total 12 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   ГОД                    205143 non-null  int64  
 1   Заявка                 205143 non-null  int64  
 2   Позиция заявки         205143 non-null  int64  
 3   Материал               204840 non-null  float64
 4   Индикатор удаления_B3  205143 non-null  bool   
 5   Группа деблокир.       162469 non-null  float64
 6   Дата заявки            205143 non-null  object 
 7   RU Наименование        204840 non-null  object 
 8   Дополнительное         34865 non-null   object 
 9   Инвентарный номер      193742 non-null  object 
 10  Закуп. организация     205143 non-null  int64  
 11  Группа материалов      205143 non-null  object 
dtypes: bool(1), float64(2), int64(4), object(5)
memory usage: 17.4+ MB


In [21]:
import pandas as pd
import re
from difflib import SequenceMatcher
from collections import Counter

# --------------------------------------------------
# 1. TEXT CLEANING
# --------------------------------------------------

def clean_text_ru(text):
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()

    # Rusça normalizasyon
    text = text.replace("ё", "е")

    # karşılaştırma operatörleri
    text = re.sub(r"[<>]=?", " ", text)

    # sayılar arasındaki nokta: 2.5 -> 2 5
    text = re.sub(r"(\d)\.(\d)", r"\1 \2", text)

    # noktalama ve özel karakterler
    text = re.sub(r"[^\w\s]", " ", text)

    # tek karakterleri kaldır
    text = re.sub(r"\b\w\b", " ", text)

    # fazla boşlukları temizle
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def tokenize(text):
    if not text:
        return []
    return text.split()

In [22]:
# --------------------------------------------------
# 2. LOTLAMA İÇİN TEMEL HAZIRLIK
# --------------------------------------------------

me5a["Дата заявки"] = pd.to_datetime(me5a["Дата заявки"], errors="coerce")

me5a["Материал"] = me5a["Материал"].apply(
    lambda x: str(int(x)) if pd.notnull(x) else ""
)

me5a["RU Наименование"] = me5a["RU Наименование"].fillna("").astype(str)
me5a["Дополнительное"] = me5a["Дополнительное"].fillna("").astype(str)
me5a["Группа материалов"] = me5a["Группа материалов"].fillna("").astype(str)
me5a["Инвентарный номер"] = me5a["Инвентарный номер"].fillna("").astype(str)

me5a["item_id"] = (
    me5a["Заявка"].astype(str).str.strip() + "_" +
    me5a["Позиция заявки"].astype(str).str.strip()
)

me5a["full_text"] = (
    me5a["RU Наименование"].str.strip() + " | " +
    me5a["Дополнительное"].str.strip() + " | " +
    me5a["Группа материалов"].str.strip()
)

me5a["clean_text"] = me5a["full_text"].apply(clean_text_ru)
me5a["tokens"] = me5a["clean_text"].apply(tokenize)

lot_df = me5a[
    [
        "item_id",
        "ГОД",
        "Заявка",
        "Позиция заявки",
        "Материал",
        "Дата заявки",
        "RU Наименование",
        "Дополнительное",
        "Инвентарный номер",
        "Группа материалов",
        "Закуп. организация",
        "full_text",
        "clean_text",
        "tokens"
    ]
].copy()

lot_df = lot_df.sort_values(
    by=["Дата заявки", "Заявка", "Позиция заявки"],
    ascending=[True, True, True]
).reset_index(drop=True)

print(lot_df.shape)
lot_df.head()

(205143, 14)


,item_id,ГОД,Заявка,Позиция заявки,Материал,Дата заявки,RU Наименование,Дополнительное,Инвентарный номер,Группа материалов,Закуп. организация,full_text,clean_text,tokens
0,2100002437_10,2024,2100002437,10,4500220423,2024-01-02,ШЛИФМАШИНА ЭКСЦЕНТРИКОВАЯ BOSCH GEX 34-150 N=...,Requester Sidorov I./644 электроинструмент,,101.5002.,2991,ШЛИФМАШИНА ЭКСЦЕНТРИКОВАЯ BOSCH GEX 34-150 N=...,шлифмашина эксцентриковая bosch gex 34 150 340...,"[шлифмашина, эксцентриковая, bosch, gex, 34, 1..."
1,2100002438_10,2024,2100002438,10,4500090953,2024-01-02,ТОПЛИВО ДИЗЕЛЬНОЕ,,,MZ-3076,2994,ТОПЛИВО ДИЗЕЛЬНОЕ | | MZ-3076,топливо дизельное mz 3076,"[топливо, дизельное, mz, 3076]"
2,2100002439_10,2024,2100002439,10,4500094636,2024-01-02,ТОВАРНЫЙ БЕТОН B30 P5 W6 D5 MAX,Годовая потребность бетона на 2024 год. ОБЯЗАТ...,,MZ-2022,2991,ТОВАРНЫЙ БЕТОН B30 P5 W6 D5 MAX | Годовая потр...,товарный бетон b30 p5 w6 d5 max годовая потреб...,"[товарный, бетон, b30, p5, w6, d5, max, годова..."
3,2100002439_20,2024,2100002439,20,4500105456,2024-01-02,ТОВАРНЫЙ БЕТОН B10 P4 W6,Годовая потребность бетона на 2024 год. ОБЯЗАТ...,,MZ-2022,2991,ТОВАРНЫЙ БЕТОН B10 P4 W6 | Годовая потребность...,товарный бетон b10 p4 w6 годовая потребность б...,"[товарный, бетон, b10, p4, w6, годовая, потреб..."
4,2100002439_30,2024,2100002439,30,4500213833,2024-01-02,БЕТОН ТОВАРНЫЙ B15 P4 W4,Годовая потребность бетона на 2024 год. ОБЯЗАТ...,,MZ-2022,2991,БЕТОН ТОВАРНЫЙ B15 P4 W4 | Годовая потребность...,бетон товарный b15 p4 w4 годовая потребность б...,"[бетон, товарный, b15, p4, w4, годовая, потреб..."


In [87]:
# --------------------------------------------------
# 3. FIXED LOT DICTIONARY
# --------------------------------------------------

fixed_lots = {
    "LOT_HILTI": {
        "type": "fixed",
        "keywords": ["hilti"],
        "update_allowed": False
    },
    "LOT_OGLAEND": {
        "type": "fixed",
        "keywords": ["oglaend"],
        "update_allowed": False
    },
    "LOT_SIKA": {
        "type": "fixed",
        "keywords": ["sika"],
        "update_allowed": False
    },
    "LOT_AKU_DOT": {
        "type": "fixed",
        "keywords": ["aku."],
        "update_allowed": False
    },
    "LOT_BETON": {
        "type": "fixed",
        "keywords": ["бетон"],
        "update_allowed": False
    },
    "LOT_КРАСКА": {
        "type": "fixed",
        "keywords": ["краск"],
        "update_allowed": False
    },
    "LOT_БОЛТ": {
        "type": "fixed",
        "keywords": ["болт","гайка","шайба"],
        "update_allowed": False
    },

}

In [24]:
# --------------------------------------------------
# 4. "aku." KONTROLÜ
# --------------------------------------------------

def matches_aku_dot_rule(row):
    raw_fields = [
        str(row.get("RU Наименование", "")).lower(),
        str(row.get("Дополнительное", "")).lower(),
        str(row.get("Инвентарный номер", "")).lower(),
        str(row.get("full_text", "")).lower()
    ]

    for field in raw_fields:
        if "aku." in field:
            return True

    return False

In [25]:
# --------------------------------------------------
# 5. FIXED LOT KONTROLÜ
# --------------------------------------------------

def check_fixed_lot(row, fixed_lots):
    # 1) Özel "aku." kuralı
    if matches_aku_dot_rule(row):
        return "LOT_AKU_DOT"

    # 2) Keyword bazlı fixed lotlar
    text = str(row.get("clean_text", "")).lower()

    for lot_id, lot_data in fixed_lots.items():
        if lot_id == "LOT_AKU_DOT":
            continue

        for kw in lot_data["keywords"]:
            if kw in text:
                return lot_id

    return None

In [26]:
# --------------------------------------------------
# 6. SKOR FONKSİYONLARI
# --------------------------------------------------

def text_similarity(a, b):
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


def keyword_overlap_score(item_tokens, lot_keywords, max_score=30):
    if not item_tokens or not lot_keywords:
        return 0.0

    item_set = set(item_tokens)
    kw_set = set(lot_keywords)

    overlap = len(item_set & kw_set)
    ratio = overlap / max(len(kw_set), 1)

    return round(min(max_score, ratio * max_score), 2)


def representative_score(item_text, rep_texts, max_score=60):
    if not item_text or not rep_texts:
        return 0.0

    sims = [text_similarity(item_text, rep) for rep in rep_texts]

    max_sim = max(sims)
    avg_sim = sum(sims) / len(sims)

    combined = 0.7 * max_sim + 0.3 * avg_sim
    return round(combined * max_score, 2)


def group_bonus_score(row, lot_data, max_score=10):
    row_org = str(row.get("Закуп. организация", "")).strip()
    lot_org = str(lot_data.get("org", "")).strip()

    if row_org and lot_org and row_org == lot_org:
        return max_score

    return 0.0


def score_learned_lot(row, lot_data):
    item_text = row["clean_text"]
    item_tokens = row["tokens"]

    kw_score = keyword_overlap_score(item_tokens, lot_data["keywords"], max_score=30)
    rep_score = representative_score(item_text, lot_data["representative_texts"], max_score=60)
    grp_score = group_bonus_score(row, lot_data, max_score=10)

    total_score = kw_score + rep_score + grp_score
    return round(total_score, 2)

In [27]:
# --------------------------------------------------
# 7. YENİ LEARNED LOT OLUŞTURMA
# --------------------------------------------------

def create_new_learned_lot(row, lot_counter):
    lot_id = f"LOT_{lot_counter:04d}"

    text = row["clean_text"]
    tokens = row["tokens"]

    lot_data = {
        "type": "learned",
        "keywords": list(dict.fromkeys(tokens[:10])),
        "anchor_text": text,
        "representative_texts": [text] if text else [],
        "update_allowed": True,
        "org": row.get("Закуп. организация", "")
    }

    return lot_id, lot_data

In [28]:
# --------------------------------------------------
# 8. LEARNED LOT GÜNCELLEME
# --------------------------------------------------

def should_learn(best_score, margin, lot_data):
    if not lot_data.get("update_allowed", False):
        return False

    if best_score >= 85 and margin >= 10:
        return True

    return False


def should_add_representative(new_text, rep_texts, similarity_threshold=0.92):
    if not new_text:
        return False

    if not rep_texts:
        return True

    sims = [text_similarity(new_text, rep) for rep in rep_texts]
    if max(sims) >= similarity_threshold:
        return False

    return True


def update_learned_lot(row, lot_data, max_reps=5, max_keywords=10):
    new_text = row["clean_text"]
    new_tokens = row["tokens"]

    # representative_texts güncelle
    if should_add_representative(new_text, lot_data["representative_texts"]):
        lot_data["representative_texts"].append(new_text)

        if len(lot_data["representative_texts"]) > max_reps:
            lot_data["representative_texts"] = lot_data["representative_texts"][:max_reps]

    # keywords güncelle
    merged = lot_data["keywords"] + new_tokens
    counts = Counter(merged)
    top_keywords = [k for k, v in counts.most_common(max_keywords)]
    lot_data["keywords"] = top_keywords

    return lot_data

In [29]:
# --------------------------------------------------
# 9. TEK KAYIT İÇİN LOT ATAMA
# --------------------------------------------------

def assign_lot_for_row(row, lots_dict, lot_counter_start):
    # 1) Fixed lot kontrolü
    fixed_lot_id = check_fixed_lot(row, fixed_lots)
    if fixed_lot_id is not None:
        return {
            "assigned_lot": fixed_lot_id,
            "decision_type": "fixed_rule",
            "best_score": 100.0,
            "second_score": 0.0,
            "margin": 100.0,
            "lot_counter": lot_counter_start
        }

    # 2) Learned lotlar arasından skorla
    learned_scores = []

    for lot_id, lot_data in lots_dict.items():
        if lot_data["type"] != "learned":
            continue

        score = score_learned_lot(row, lot_data)
        learned_scores.append((lot_id, score))

    # 3) Hiç learned lot yoksa yeni lot aç
    if len(learned_scores) == 0:
        new_lot_id, new_lot_data = create_new_learned_lot(row, lot_counter_start)
        lots_dict[new_lot_id] = new_lot_data

        return {
            "assigned_lot": new_lot_id,
            "decision_type": "new_lot_created",
            "best_score": 0.0,
            "second_score": 0.0,
            "margin": 0.0,
            "lot_counter": lot_counter_start + 1
        }

    learned_scores = sorted(learned_scores, key=lambda x: x[1], reverse=True)

    best_lot, best_score = learned_scores[0]
    second_score = learned_scores[1][1] if len(learned_scores) > 1 else 0.0
    margin = round(best_score - second_score, 2)

    # 4) Karar
    if best_score >= 75:
        # Learned lota ata
        if should_learn(best_score, margin, lots_dict[best_lot]):
            lots_dict[best_lot] = update_learned_lot(row, lots_dict[best_lot])

        return {
            "assigned_lot": best_lot,
            "decision_type": "assigned_existing_learned",
            "best_score": best_score,
            "second_score": second_score,
            "margin": margin,
            "lot_counter": lot_counter_start
        }

    elif 60 <= best_score < 75:
        # Şimdilik review yerine yine lot ata ama flag koy
        return {
            "assigned_lot": best_lot,
            "decision_type": "review_needed",
            "best_score": best_score,
            "second_score": second_score,
            "margin": margin,
            "lot_counter": lot_counter_start
        }

    else:
        # Yeni learned lot aç
        new_lot_id, new_lot_data = create_new_learned_lot(row, lot_counter_start)
        lots_dict[new_lot_id] = new_lot_data

        return {
            "assigned_lot": new_lot_id,
            "decision_type": "new_lot_created",
            "best_score": best_score,
            "second_score": second_score,
            "margin": margin,
            "lot_counter": lot_counter_start + 1
        }

In [30]:
# --------------------------------------------------
# 10. TÜM DATAFRAME İÇİN LOTLAMA
# --------------------------------------------------

def process_me5a_lots(lot_df, fixed_lots):
    lots_dict = {}

    # fixed lotları başlangıçta dictionary'ye koy
    for lot_id, lot_data in fixed_lots.items():
        lots_dict[lot_id] = lot_data.copy()

    results = []
    lot_counter = 1

    for idx, row in lot_df.iterrows():
        result = assign_lot_for_row(row, lots_dict, lot_counter)
        lot_counter = result["lot_counter"]

        results.append({
            "item_id": row["item_id"],
            "Заявка": row["Заявка"],
            "Позиция заявки": row["Позиция заявки"],
            "Дата заявки": row["Дата заявки"],
            "RU Наименование": row["RU Наименование"],
            "Дополнительное": row["Дополнительное"],
            "Инвентарный номер": row["Инвентарный номер"],
            "clean_text": row["clean_text"],
            "assigned_lot": result["assigned_lot"],
            "decision_type": result["decision_type"],
            "best_score": result["best_score"],
            "second_score": result["second_score"],
            "margin": result["margin"]
        })

    result_df = pd.DataFrame(results)
    return result_df, lots_dict

In [68]:
def process_me5a_lots(lot_df, fixed_lots):
    lots_dict = {}

    for lot_id, lot_data in fixed_lots.items():
        lots_dict[lot_id] = lot_data.copy()

    results = []
    lot_counter = 1
    total_rows = len(lot_df)

    for idx, row in lot_df.iterrows():
        if idx % 5000 == 0:
            print(f"İşlenen satır: {idx} / {total_rows}")

        result = assign_lot_for_row(row, lots_dict, lot_counter)
        lot_counter = result["lot_counter"]

        results.append({
            "item_id": row["item_id"],
            "Заявка": row["Заявка"],
            "Позиция заявки": row["Позиция заявки"],
            "Дата заявки": row["Дата заявки"],
            "RU Наименование": row["RU Наименование"],
            "Дополнительное": row["Дополнительное"],
            "Инвентарный номер": row["Инвентарный номер"],
            "clean_text": row["clean_text"],
            "assigned_lot": result["assigned_lot"],
            "decision_type": result["decision_type"],
            "best_score": result["best_score"],
            "second_score": result["second_score"],
            "margin": result["margin"]
        })

    result_df = pd.DataFrame(results)
    return result_df, lots_dict

In [74]:
sample_df = lot_df.head(20000).copy()

result_df, final_lots_dict = process_me5a_lots(sample_df, fixed_lots)

print(result_df.shape)
result_df.head(20)

İşlenen satır: 0 / 20000
İşlenen satır: 5000 / 20000
İşlenen satır: 10000 / 20000
İşlenen satır: 15000 / 20000
(20000, 13)


,item_id,Заявка,Позиция заявки,Дата заявки,RU Наименование,Дополнительное,Инвентарный номер,clean_text,assigned_lot,decision_type,best_score,second_score,margin
0,2100002437_10,2100002437,10,2024-01-02,ШЛИФМАШИНА ЭКСЦЕНТРИКОВАЯ BOSCH GEX 34-150 N=...,Requester Sidorov I./644 электроинструмент,,шлифмашина эксцентриковая bosch gex 34 150 340...,LOT_0001,new_lot_created,0.00,0.0,0.00
1,2100002438_10,2100002438,10,2024-01-02,ТОПЛИВО ДИЗЕЛЬНОЕ,,,топливо дизельное mz 3076,LOT_0002,new_lot_created,7.27,0.0,7.27
2,2100002439_10,2100002439,10,2024-01-02,ТОВАРНЫЙ БЕТОН B30 P5 W6 D5 MAX,Годовая потребность бетона на 2024 год. ОБЯЗАТ...,,товарный бетон b30 p5 w6 d5 max годовая потреб...,LOT_BETON,fixed_rule,100.00,0.0,100.00
3,2100002439_20,2100002439,20,2024-01-02,ТОВАРНЫЙ БЕТОН B10 P4 W6,Годовая потребность бетона на 2024 год. ОБЯЗАТ...,,товарный бетон b10 p4 w6 годовая потребность б...,LOT_BETON,fixed_rule,100.00,0.0,100.00
4,2100002439_30,2100002439,30,2024-01-02,БЕТОН ТОВАРНЫЙ B15 P4 W4,Годовая потребность бетона на 2024 год. ОБЯЗАТ...,,бетон товарный b15 p4 w4 годовая потребность б...,LOT_BETON,fixed_rule,100.00,0.0,100.00
5,2100002439_40,2100002439,40,2024-01-02,ТОВАРНЫЙ БЕТОН B20 P4 W6,Годовая потребность бетона на 2024 год. ОБЯЗАТ...,,товарный бетон b20 p4 w6 годовая потребность б...,LOT_BETON,fixed_rule,100.00,0.0,100.00
6,2100002439_50,2100002439,50,2024-01-02,ТОВАРНЫЙ БЕТОН B25 P4 W6,Годовая потребность бетона на 2024 год. ОБЯЗАТ...,,товарный бетон b25 p4 w6 годовая потребность б...,LOT_BETON,fixed_rule,100.00,0.0,100.00
7,2100002439_60,2100002439,60,2024-01-02,ТОВАРНЫЙ БЕТОН B25 P4 W8,Годовая потребность бетона на 2024 год. ОБЯЗАТ...,,товарный бетон b25 p4 w8 годовая потребность б...,LOT_BETON,fixed_rule,100.00,0.0,100.00
8,2100002439_70,2100002439,70,2024-01-02,БЕТОН ТОВАРНЫЙ B25 P4 W4,Годовая потребность бетона на 2024 год. ОБЯЗАТ...,,бетон товарный b25 p4 w4 годовая потребность б...,LOT_BETON,fixed_rule,100.00,0.0,100.00
9,2100002439_80,2100002439,80,2024-01-02,ТОВАРНЫЙ БЕТОН B30 P6 W10 SF2 230KG,Годовая потребность бетона на 2024 год. ОБЯЗАТ...,,товарный бетон b30 p6 w10 sf2 230kg годовая по...,LOT_BETON,fixed_rule,100.00,0.0,100.00


In [72]:
result_df, final_lots_dict = process_me5a_lots(lot_df, fixed_lots)

print(result_df.shape)
result_df.head(20)

İşlenen satır: 0 / 205143
İşlenen satır: 5000 / 205143
İşlenen satır: 10000 / 205143
İşlenen satır: 15000 / 205143
İşlenen satır: 20000 / 205143
İşlenen satır: 25000 / 205143
İşlenen satır: 30000 / 205143
İşlenen satır: 35000 / 205143
İşlenen satır: 40000 / 205143
İşlenen satır: 45000 / 205143
İşlenen satır: 50000 / 205143
İşlenen satır: 55000 / 205143
İşlenen satır: 60000 / 205143


KeyboardInterrupt: 

In [75]:
result_df["decision_type"].value_counts()

,count
decision_type,
fixed_rule,19085
assigned_existing_learned,440
review_needed,238
new_lot_created,237


In [76]:
result_df["assigned_lot"].value_counts().head(20)

,count
assigned_lot,
LOT_AKU_DOT,18919
LOT_0099,108
LOT_BETON,106
LOT_0191,83
LOT_HILTI,60
LOT_0021,49
LOT_0214,37
LOT_0028,35
LOT_0002,28


In [84]:
result_df[["RU Наименование","assigned_lot","best_score"]].sample(n=5)

,RU Наименование,assigned_lot,best_score
16850,НУТРОМЕР МИКРОСКОПИЧЕСКИЙ НМ-5-30 ЦЕНА ДЕЛЕНИ...,LOT_0180,59.5
16054,10UJA.KAA.TM.TB0102-22/27.М194-1,LOT_AKU_DOT,100.0
8122,КРЫШКА SPB CO-CT-600-3000 HDG C4 АРТ.121581 Ф....,LOT_AKU_DOT,100.0
9222,AKU.0120.00USY.0.KM.LC0006.1CRB-6,LOT_AKU_DOT,100.0
10279,AKU.0120.00USY.0.KM.LC0019.CRB2/31,LOT_AKU_DOT,100.0


In [86]:
result_df.to_excel("/content/me5a_lot_result.xlsx", index=False)